# Altimetry Search - Search for Swot passes

This notebook is similar to the notebook [main](main.ipynb) but it uses the the
public API of the module to search for SWOT passes using the programmatic
interface instead of the
[voila](https://voila.readthedocs.io/en/stable/using.html) interface.

First, we import the module and create the application object. It is not
mandatory to create the application object but it is useful to have access to
the GUI to create the bounding box.

In [ ]:
from altimetry.search.gui import MapSelection, compute_selected_passes

In [ ]:
app = MapSelection()

Display the GUI to create the bounding box.

In [ ]:
app.display()

The easiest way to get the selected passes is to use the method provided by the application.

In [ ]:
from pyinterp.geometry import geographic
from altimetry.search import Mission
import warnings

if app.selection is None:
    warnings.warn('No selection has been done with the GUI, '
                  'a default selection is used')
    app.selection = geographic.algorithms.from_wkt(
        'POLYGON((-48 -12,-48 42,0 42,0 -12,-48 -12))')
if app.mission_widget.value is None:    
    app.mission_widget.value = Mission.SWOT_SWATH_SCIENCE

first_date, search_duration = app.date_selection.values()

df = compute_selected_passes(
    app.selection, 
    first_date, 
    search_duration,
    app.mission_widget.value
)
df

To save the result in a CSV file execute: `df.to_csv("passes.csv")`

## API

In [ ]:
import numpy
from pyinterp.geometry import geographic
from altimetry.search.gui import load_polygons
from altimetry.search import Mission, get_pass_passage_time, get_selected_passes, get_passes_crossing_polygon

In [ ]:
mission = Mission.SWOT_SWATH_SCIENCE

### Get half-orbits from a time period

An other way to get the selected passes is to use the function provided by the package.

Using the `get_selected_passes` function, the following code computes the selected passes for the next 72 hours. The first
parameter is the current time, the second parameter is the time interval.

*Adjust the parameters as needed.*

In [ ]:
selected_passes = get_selected_passes(mission, numpy.datetime64('now'), numpy.timedelta64(3, 'D'))
selected_passes

### Get the passes intersecting a polygon

Here you have two options:

1. Use the polygon defined in the app : `app.selection`
2. Define your own polygon, using the
   [WKT](https://en.wikipedia.org/wiki/Well-known_text_representation_of_geometry)
   format.
3. Define your own polygon, using [GeoJSON](https://en.wikipedia.org/wiki/GeoJSON)

In [ ]:
bbox = geographic.algorithms.from_wkt(
    'POLYGON((-6 36,-6 60,36 60,36 36,-6 36))')

The function `get_passes_crossing_polygon` supports two distinct use cases: :

#### 1- No pass numbers known yet: get every pass of `mission`'s orbit that falls in `polygon`.

In [ ]:
get_passes_crossing_polygon(mission=mission, polygon=bbox)

#### 2- A candidate subset of pass numbers is already known (e.g. from `get_selected_passes`): pass it as `passes` to eliminate those that do not fall in `polygon`, keeping only the ones that do

In [ ]:
passes = numpy.array(sorted(set(selected_passes['pass_number'])))
passes

In [ ]:
get_passes_crossing_polygon(mission=mission, polygon=bbox, passes=passes)

### Get the passage time of passes intersecting a given polygon

Now we compute the passage time for the selected passes for a given polygon using `get_pass_passage_time`.

In [ ]:
import pandas

pass_passage_time = pandas.DataFrame(get_pass_passage_time(mission, selected_passes=selected_passes, polygon=bbox))
pass_passage_time

## To display the SWATH with matplotlib

The last section of this notebook shows how to plot the selected passes with
matplotlib.

In [ ]:
left_swath, right_swath = load_polygons(mission, pass_passage_time['pass_number'].values)

In [ ]:
import cartopy.crs
import matplotlib.pyplot
import matplotlib
import matplotlib.patches
import matplotlib.colors
import matplotlib.cm

color_norm = matplotlib.colors.Normalize(vmin=0,
                                         vmax=pass_passage_time['pass_number'].values.max())
color_map = matplotlib.cm.ScalarMappable(norm=color_norm, cmap='jet')

fig = matplotlib.pyplot.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, projection=cartopy.crs.PlateCarree())
ax.coastlines()
ax.gridlines()

points = bbox.outer
lons = numpy.array([p.lon for p in points])
lats = numpy.array([p.lat for p in points])
labels = {}

ax.plot(lons, lats, transform=cartopy.crs.Geodetic(), color='red')

wgs84 = geographic.Spheroid()

for pass_number, item in left_swath:
    intersection_list = geographic.algorithms.intersection(
            item, bbox,
            spheroid=wgs84)
    if len(intersection_list) == 0:
        continue
    outer = intersection_list[0].outer
    poly = matplotlib.patches.Polygon([(p.lon, p.lat) for p in outer],
                                      transform=cartopy.crs.PlateCarree(),
                                      facecolor=color_map.to_rgba(pass_number),
                                      alpha=0.5)
    labels[pass_number] = True
    poly.set_label(f'Pass {pass_number}')
    ax.add_patch(poly)
    
for pass_number, item in right_swath:
    intersection_list = geographic.algorithms.intersection(
            item, bbox,
            spheroid=wgs84)
    if len(intersection_list) == 0:
        continue
    outer = intersection_list[0].outer
    poly = matplotlib.patches.Polygon([(p.lon, p.lat) for p in outer],
                                      transform=cartopy.crs.PlateCarree(),
                                      facecolor=color_map.to_rgba(pass_number),
                                      alpha=0.5)
    if pass_number not in labels:
        poly.set_label(f'Pass {pass_number}')
    ax.add_patch(poly)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc='lower left')